# 

title: Water Rights Restored to the Gila River

subtitle: The impacts of irrigation on vegetation health in the Gila
River Valley

author:

-   Elsa Culler

-   Nate Quarderer

date: last-modified

image: /img/earth-analytics/water-rights/lesson-water-rights.png

image-alt: “Dry river with dead plants turns into a stream with living
plants”

description: \|

In 2004, the Akimel O’‘otham and Tohono O’’odham tribes won a water
rights settlement in the US Supreme Court. Using satellite imagery, we
can see the effects of irrigation water on the local vegetation.

learning-goals:

-   Open raster or image data using code

-   Combine raster data and vector data to crop images to an area of
    interest

-   Summarize raster values with stastics

-   Analyze a time-series of raster images

params:

id: stars

site_name: Gila River Indian Community

event: water rights case

data_dir: gila-river

jupyter:

kernelspec:

    name: learning-portal

    language: python

    display_name: Learning Portal

# STEP 0: Set up

To get started on this notebook, you’ll need to restore any variables
from previous notebooks to your workspace. To save time and memory, make
sure to specify which variables you want to load.

In [1]:
%store -r Gila_boundary_gdf aitsn_gdf ndvi_da ndvi_diff_da ndvi_diff_scaled

no stored variable or alias ndvi_diff_da
no stored variable or alias ndvi_diff_scaled


You will also need to import any libraries you are using in this
notebook, since they won’t carry over from the previous notebook:

In [2]:
# Import libraries
import earthpy 
import xarray as xr
import rioxarray as rxr
import hvplot.xarray
import hvplot.pandas
from shapely.geometry import mapping
import matplotlib.pyplot as plt



# STEP 4: Is the NDVI different within the **?meta:params.site_name** after the **?meta:params.event**?

You will compute the mean NDVI inside and outside the Gila River Indian Community boundary.
First, use the code below to get a `GeoDataFrame` of the area outside
the Reservation.

<link rel="stylesheet" type="text/css" href="./assets/styles.css"><div class="callout callout-style-default callout-titled callout-task"><div class="callout-header"><div class="callout-icon-container"><i class="callout-icon"></i></div><div class="callout-title-container flex-fill">Try It</div></div><div class="callout-body-container callout-body"><ol type="1">
<li>Check the variable names - Make sure that the code uses your
boundary <code>GeoDataFrame</code></li>
<li>How could you test if the geometry was modified correctly? Add some
code to take a look at the results.</li>
</ol></div></div>

In [ ]:
# Compute the area outside reservation
# I didn't make a new shapefile because I used the 'invert = True' argument in my .clip function.
# This clipped NDVI to everything outside the Gila River Valley polygon

Next, clip your DataArray to the boundaries for both inside and outside
the reservation. You will need to replace the `GeoDataFrame` name with
your own. Check out the [lesson on clipping data with the `rioxarray`
library in the
textbook](https://www.earthdatascience.org/courses/use-data-open-source-python/intro-raster-data-python/raster-data-processing/crop-raster-data-with-shapefile-in-python/).

> **GOTCHA ALERT**
>
> It’s important to use `from_disk=True` when clipping large arrays like
> this. It allows the computer to use less valuable memory resources
> when clipping - you will probably find that otherwise the cell below
> crashes your kernel.

In [ ]:
# Clip data to both inside and outside the boundary
ndvi_clipped_in = ndvi_da.rio.clip(Gila_boundary_gdf.geometry.apply(mapping),
                                      Gila_boundary_gdf.crs, # matching CRS 
                                      from_disk = True) # uses less memory when clipping 




In [ ]:
ndvi_clipped_out = ndvi_da.rio.clip(Gila_boundary_gdf.geometry.apply(mapping),
                                      Gila_boundary_gdf.crs, # matching CRS
                                      from_disk = True, # uses less memory when clipping 
                                      invert = True) # clip to everything outside the polygon 



<link rel="stylesheet" type="text/css" href="./assets/styles.css"><div class="callout callout-style-default callout-titled callout-task"><div class="callout-header"><div class="callout-icon-container"><i class="callout-icon"></i></div><div class="callout-title-container flex-fill">Try It</div></div><div class="callout-body-container callout-body"><p>For <strong>both inside and outside</strong> the <span
data-__quarto_custom="true" data-__quarto_custom_type="Shortcode"
data-__quarto_custom_context="Inline"
data-__quarto_custom_id="3"></span> boundary:</p>
<ul>
<li>Group the data by year</li>
<li>Take the mean. You always need to tell reducing methods in
<code>xarray</code> what dimensions you want to reduce. When you want to
summarize data across <strong>all</strong> dimensions, you can use the
<code>...</code> syntax, e.g. <code>.mean(...)</code> as a
shorthand.</li>
<li>Select the NDVI variable</li>
<li>Convert to a DataFrame using the <code>to_dataframe()</code>
method</li>
<li>Join the two DataFrames for plotting using the <code>.join()</code>
method. You will need to rename the columns using the
<code>lsuffix=</code> and <code>rsuffix=</code> parameters</li>
</ul>
<p>Finally, plot annual July means for both inside and outside the
Reservation on the same plot.</p></div></div>

> **GOTCHA ALERT**
>
> The DateIndex in pandas is a little different from the Datetime
> Dimension in xarray. You will need to use the `.dt.year` syntax to
> access information about the year, not just `.year`.

In [ ]:
# Get all July NDVI values
ndvi_in_july = ndvi_clipped_in.sel(date=ndvi_clipped_in['date'].dt.month == 7)
ndvi_out_july = ndvi_clipped_out.sel(date=ndvi_clipped_out['date'].dt.month == 7)

# Calculate mean July values
annual_in = ndvi_in_july.groupby('date.year').mean(dim='date').mean(dim=['x','y'])
annual_out = ndvi_out_july.groupby('date.year').mean(dim='date').mean(dim=['x','y'])

da_in = annual_in['NDVI'] # select only NDVI
df_in = da_in.to_dataframe(name='NDVI_inside') # make into dataframe
df_in = df_in.drop(columns=['band', 'spatial_ref']) # drop unnecessary columns

da_out = annual_out['NDVI']  
df_out = da_out.to_dataframe(name='NDVI_outside')
df_out = df_out.drop(columns=['band', 'spatial_ref'])

# make one dataframe with both inside and outside values
df_combined = df_in.join(df_out, how='outer')

In [ ]:
df_plot = df_combined.copy() # copy of df just in case

# scale NDVI
df_plot['NDVI_inside'] = df_plot['NDVI_inside'] * 0.0001 
df_plot['NDVI_outside'] = df_plot['NDVI_outside'] * 0.0001

# Plot
df_plot.hvplot(x='year', y=['NDVI_inside', 'NDVI_outside'], 
               ylabel='NDVI', title='NDVI Inside and Outside The Gila Reservation')

:NdOverlay   [Variable]
   :Curve   [year]   (value)

Now, take the difference between outside and inside the site boundary
and plot that. What do you observe? Don’t forget to write a headline and
description of your plot!

In [ ]:
# Plot difference inside and outside the boundary

df_diff = df_plot.copy() # copy again just in case

# calculate difference between inside & outside NDVI
df_diff['NDVI_diff'] =  df_plot['NDVI_inside'] - df_plot['NDVI_outside'] 


df_diff.hvplot(x='year', y='NDVI_diff',
                   ylabel='NDVI Difference',
                   title='Difference in Annual July NDVI: Outside vs Inside Gila Boundary',
                   line_color='steelblue')

:Curve   [year]   (NDVI_diff)

Headline:

A Gap Persists in Vegetation Health Between the Gila River Valley and Surrounding Areas, but Increased Water Access Is Driving Recovery 

Description:

Vegetation health remains lower within the Gila River Valley than in surrounding areas, as shown by consistently negative NDVI difference values (calculated as inside-outside). This indicates that vegetation outside the reservation continues to be healthier overall. However, since 2004, when the Akimel O’otham and Tohono O’odham tribes regained rights to their water in the Gila River Valley, vegetation within the boundary has shown gradual improvement. The upward trend in the NDVI difference suggests that conditions inside the boundary are becoming increasingly similar to those outside, reflecting a slow but encouraging recovery in vegetation health.

# STEP -1: Wrap up

Don’t forget to store your variables so you can use them in other
notebooks! Replace `var1` and `var2` with the variable you want to save,
separated by spaces.

In [11]:
%store var1 var2

Finally, be sure to `Restart` and `Run all` to make sure your notebook
works all the way through!